## Setup

In [ ]:
import pandas as pd  # tables of data
import statsmodels.formula.api as smf  # regressions written as formulas

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive
survey_year = 2026  # which survey year the regression uses

## Mount Google Drive

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## Prepare the sample

In [ ]:
x_vars = ["age", "female", "black", "asian", "hispanic", "other_race", "hs_ged", "some_college", "bachelor_plus"]  # independent variables; comparison groups: male, White, less than high school

sample = df[df["year"] == survey_year].copy()  # keep only the chosen survey year
sample["neet100"] = sample["neet"] * 100  # NEET in percent (0 or 100), so coefficients are percentage points

model_vars = ["neet100"] + x_vars  # every variable used in the regression
left_out = sample[model_vars].isna().any(axis=1).sum()  # people with any of these missing
model_data = sample.dropna(subset=model_vars)  # leave them out

print("Survey year:", survey_year)  # which year
print("People in the survey year:", len(sample))  # before leaving anyone out
print("People left out because a regression variable is missing (unweighted):", left_out)  # how many were left out
print("People in the regression:", len(model_data))  # after leaving them out

## Weighted NEET rates by group

In [ ]:
def wmean(data, var):  # weighted mean of one variable
    d = data.dropna(subset=[var])  # leave out people with a missing value
    return (d[var] * d["asecwt"]).sum() / d["asecwt"].sum()  # sum of value x weight, divided by sum of weights

groups = {  # row label: (variable, value) that puts a person in the row; None means everyone
    "All people": None,  # everyone in the regression
    "Men": ("female", 0),  # female = 0
    "Women": ("female", 1),  # female = 1
    "White, non-Hispanic": ("white", 1),  # white dummy = 1
    "Black, non-Hispanic": ("black", 1),  # black dummy = 1
    "Asian, non-Hispanic": ("asian", 1),  # asian dummy = 1
    "Hispanic (any race)": ("hispanic", 1),  # hispanic dummy = 1
    "Other, non-Hispanic": ("other_race", 1),  # other_race dummy = 1
    "Less than high school": ("less_hs", 1),  # less_hs dummy = 1
    "High school or GED": ("hs_ged", 1),  # hs_ged dummy = 1
    "Some college": ("some_college", 1),  # some_college dummy = 1
    "Bachelor or higher": ("bachelor_plus", 1),  # bachelor_plus dummy = 1
}  # all the groups in the regression

rows = {}  # one row per group
for label, condition in groups.items():  # loop over the groups
    group = model_data if condition is None else model_data[model_data[condition[0]] == condition[1]]  # people in this group
    rows[label] = {"NEET rate (%)": wmean(group, "neet100"), "N": len(group)}  # weighted rate and unweighted respondents

table = pd.DataFrame(rows).T  # dictionary -> table, one row per group
table["N"] = table["N"].astype(int)  # N is a whole number
print(f"Table. NEET rate by group, % of people, survey year {survey_year} (weighted with asecwt).")  # table title
print(table.to_string(float_format="{:.1f}".format))  # table, 1 decimal

## Regression

In [ ]:
formula = "neet100 ~ " + " + ".join(x_vars)  # neet100 ~ age + female + black + ...
model = smf.wls(formula, data=model_data, weights=model_data["asecwt"]).fit(cov_type="HC1")  # weighted OLS with asecwt, robust standard errors

def stars(p):  # significance stars for one p-value
    if p < 0.01:  # very small p-value
        return "***"  # 1% level
    if p < 0.05:  # small p-value
        return "**"  # 5% level
    if p < 0.1:  # fairly small p-value
        return "*"  # 10% level
    return ""  # not significant

labels = {  # variable names -> labels, in table order
    "age": "Age (years)",  # age in years
    "female": "Female",  # 1 = Women, 0 = Men
    "black": "Black",  # compared with White
    "asian": "Asian",  # compared with White
    "hispanic": "Hispanic",  # compared with White
    "other_race": "Other race",  # compared with White
    "hs_ged": "High school or GED",  # compared with less than high school
    "some_college": "Some college",  # compared with less than high school
    "bachelor_plus": "Bachelor or higher",  # compared with less than high school
    "Intercept": "Constant",  # the constant
}  # end of labels

results = pd.DataFrame({"Coefficient": model.params.map("{:.2f}".format) + model.pvalues.apply(stars).str.ljust(3),  # coefficient + stars, as text
                        "Robust std. error": model.bse.map("{:.2f}".format)})  # standard error, as text
results = results.loc[list(labels)].rename(index=labels)  # rows in our order, with labels

print(f"NEET (percentage points). Survey year {survey_year}, ages 16-29, weighted with asecwt.")  # table title
print(results.to_string())  # coefficient table
print("N =", f"{int(model.nobs):,}", "| R-squared =", f"{model.rsquared:.2f}")  # sample size and R-squared
print("Comparison group: male, White, less than high school.")  # comparison groups
print("* p<0.10, ** p<0.05, *** p<0.01 (robust standard errors)")  # key for the stars